In [38]:
import os
import json
from tqdm import tqdm

In [39]:
# Base ID starts from 2500 for all data points

In [40]:
MULTITURN_DATA="/proj/m3benchmark/m3data/0905/m3_train_test_ood_rest_v2"
API_FOLDERNAME="/proj/m3benchmark/invocable-api-datasets/rest_train/DGT_enriched/v2_0809"
OUTPUT_FOLDERNAME="/proj/m3benchmark/m3data/0905/m3_train_test_ood_rest_v2/single_turn"

In [41]:
def get_atomic_question(data):
    atomic_question=[]
    for item in data:
        for turn in item["turns"]:
            for hop in turn["gold_sequence"]:
                if hop["question_type"]=='API':
                    atomic_question.append(hop["question"])
    return set(atomic_question)

In [42]:
def combine_data(domain_name):
    data=[]

    train_path=f"{MULTITURN_DATA}/train/{domain_name}_multiturn_bird.json"
    if os.path.exists(train_path) and os.path.isfile(train_path):
        with open(train_path, 'r') as j:
            data.extend(json.load(j))

    test_path=f"{MULTITURN_DATA}/test/{domain_name}_multiturn_bird.json"
    if os.path.exists(test_path) and os.path.isfile(test_path):
        with open(f"{MULTITURN_DATA}/test/{domain_name}_multiturn_bird.json", 'r') as j:
            data.extend(json.load(j))

    ood_path=f"{MULTITURN_DATA}/ood/{domain_name}_multiturn_bird.json"
    if os.path.exists(ood_path) and os.path.isfile(ood_path):
        with open(f"{MULTITURN_DATA}/ood/{domain_name}_multiturn_bird.json", 'r') as j:
            data.extend(json.load(j))
    return data

In [43]:
def get_turn(api_elem):
    turn={
        "query": api_elem["input"],
        "answer": api_elem["gold_answer"],
        "type": "(API)",
        "gold_sequence": [{
            "sample_id": api_elem["sample_id"],
            "db_id": api_elem["dataset_name"],
            "question": api_elem["input"],
            "evidence": "",
            "SQL": api_elem["query"],
            "answer": api_elem["gold_answer"],
            "parameters": [],
            "links": {},
            "direct_links": [],
            "question_type": "API",
            "OUTPUT_AFTER_EXECUTING_API": api_elem["OUTPUT_AFTER_EXECUTING_API"],
            "API": api_elem["API"],
            "output": api_elem["output"],
            "ignore": api_elem["ignore"],
            "parameterized_query": api_elem["parameterized_query"]
        }]
    }
    return [turn]

In [44]:
def make_multiturn_elem(api_elem, idx):
    item = {
        "task_name": api_elem["dataset_name"],
        "is_seed": False,
        "sample_id": idx,
        "dataset_name": api_elem["dataset_name"],
        "turns": get_turn(api_elem),
        "num_turns": 1,
        "num_hops": [
            1
        ],
        "type": "(API)",
        "retrievers": [],
        "tools": api_elem["tools"]
    }
    return item

In [49]:
DOMAINS=["address", "airline", "app_store", "authors", "beer_factory", "bike_share_1", "book_publishing_company", "books", "cars", "chicago_crime", "citeseer", "codebase_comments", "coinmarketcap", "college_completion", "computer_student", "cookbook", "disney", "european_football_1", "food_inspection", "genes", "hockey", "ice_hockey_draft", "image_and_language", "law_episode", "mental_health_survey", "menu", "mondial_geo", "movielens", "movie", "movies_4", "music_tracker", "olympics", "professional_basketball", "public_review_platform", "restaurant", "sales_in_weather", "shakespeare", "simpson_episodes", "soccer_2016", "student_loan", "talkingdata", "trains", "university", "video_games", "world_development_indicators", "world",'california_schools',
 'card_games','codebase_community','debit_card_specializing','european_football_2','financial','formula_1','student_club','superhero','thrombosis_prediction','toxicology', 'music_platform_2','language_corpus', 'donor','legislator', 'movie_platform','craftbeer',"movie_3"]

In [50]:
len(DOMAINS)

64

In [53]:
idx=2500
for domain in tqdm(DOMAINS):
    single_turn_data=[]
    data=combine_data(domain)
    api_path=f"{API_FOLDERNAME}/train/en_mixtral-8x22b_{domain}_nestful_format_bird.json"
    if os.path.exists(api_path) and os.path.isfile(api_path):
        with open(api_path, 'r') as j:
            api_data = json.load(j)
    elif os.path.exists(f"{API_FOLDERNAME}/dev/en_mixtral-8x22b_{domain}_nestful_format_bird.json") and os.path.isfile(f"{API_FOLDERNAME}/dev/en_mixtral-8x22b_{domain}_nestful_format_bird.json"):
        with open(f"{API_FOLDERNAME}/dev/en_mixtral-8x22b_{domain}_nestful_format_bird.json", 'r') as j:
            api_data = json.load(j)
    else:
        import pdb
        pdb.set_trace()

    atomic_questions=get_atomic_question(data)
    for api_item in api_data:
        if not api_item["ignore"]:
            if api_item["input"] not in atomic_questions:
                single_turn_data.append(make_multiturn_elem(api_item, idx))
                idx+=1
    with open(f'{OUTPUT_FOLDERNAME}/{domain}_multiturn_bird.json', 'w') as fout:
        json.dump(single_turn_data , fout)

100%|██████████| 64/64 [04:11<00:00,  3.93s/it]


In [54]:
idx-2500

4288

In [ ]:
dev_domain=["california_schools","card_games","codebase_community","debit_card_specializing","european_football_2","financial"]

In [36]:
# All domains in BIRD dataset
all_domains=["california_schools", "card_games", "codebase_community", "debit_card_specializing", "european_football_2", "financial", "formula_1", "student_club", "superhero", "thrombosis_prediction", "toxicology"
            ,"music_platform_2", "shooting", "car_retails", "airline", "human_resources", "student_loan", "codebase_comments", "language_corpus", "bike_share_1", "cookbook", "software_company", "donor", "authors"
            , "shipping", "video_games", "sales", "olympics", "university", "talkingdata", "simpson_episodes", "movielens", "mondial_geo", "legislator", "regional_sales", "world_development_indicators", "food_inspection_2"
            , "retail_world", "citeseer", "computer_student", "college_completion", "synthea", "book_publishing_company", "trains", "retails", "soccer_2016", "law_episode", "food_inspection", "european_football_1", "mental_health_survey"
            , "hockey", "public_review_platform", "retail_complains", "ice_hockey_draft", "menu", "cs_semester", "beer_factory", "cars", "genes", "shakespeare", "image_and_language", "disney", "music_tracker", "works_cycles"
            , "movie_platform", "books", "social_media", "restaurant", "superstore", "address", "chicago_crime", "professional_basketball", "coinmarketcap", "movies_4", "sales_in_weather", "app_store", "craftbeer", "movie", "world","movie_3"]

# Domains to exclude for personal information
red_domains=["car_retails", "synthea", "shipping", "cs_semester", "food_inspection_2"
                , "sales", "software_company", "social_media", "human_resources", "regional_sales"
                ,"works_cycles", "retails", "retail_world", "retail_complains", "shooting", "superstore"]

In [37]:
len(all_domains)-len(red_domains)

64

In [35]:
unused_domains=[]
for dom in all_domains:
    if (dom in red_domains) and (dom in DOMAINS):
        print(dom)

movie_3


In [47]:
unused_domains=[]
for dom in all_domains:
    if (dom not in red_domains) and (dom not in DOMAINS):
        unused_domains.append(dom)

In [48]:
unused_domains

['movie_3']

In [25]:
len(DOMAINS)

64